# Step 1 — Targeted EDA: monthly variables vs. target

Before implementing `build_monthly_trend_features()` in `src/features.py`, we need to
validate with real data whether the monthly trends (m1 -> m6) actually relate to
`liquidity_stress_next_30d`. See `RESOURCES.md` for why (lesson from Home Credit:
well-targeted feature engineering matters more than model tuning).

This notebook uses the functions from `src/` instead of rewriting the data loading,
as recommended in the README.

**Cell 1**: imports and data loading.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config, data

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 200)

train, test = data.load_raw_data()
print("train:", train.shape, "| test:", test.shape)
train.head()

**Cell 2**: group the monthly columns by family (`m1_..m6_`) to find out
how many trend variables need to be built and confirm that all 6 months
exist for every family.

In [ ]:
import re

monthly_cols = [c for c in train.columns if re.match(r"^m[1-6]_", c)]
col_set = set(monthly_cols)

# strip the m1_..m6_ prefix to group by variable family
families = sorted(set(re.sub(r"^m[1-6]_", "", c) for c in monthly_cols))

print(f"{len(monthly_cols)} monthly columns -> {len(families)} variable families\n")
for fam in families:
    months_present = [m for m in range(1, 7) if f"m{m}_{fam}" in col_set]
    print(f"{fam:35s} months present: {months_present}")

**Cell 3**: compare how each family evolves chronologically — from 6 months ago
(`m6`, oldest) up to the most recent month (`m1`) — split by target. Per
`data_dictionary.csv`, **m1 is the most recent month and m6 is the oldest**, so
the x-axis is plotted in reverse (`ax.invert_xaxis()`) to read left-to-right as
oldest -> most recent, like a normal time series. We're looking to see whether
customers with `liquidity_stress_next_30d = 1` show a different trend (e.g.
average balance dropping, withdrawals rising) as they approach the prediction
point, compared to those who don't.

In [ ]:
key_families = ["daily_avg_bal", "withdraw_total_value", "received_total_value"]

# 1 row x 3 columns of subplots, one per variable family
fig, axes = plt.subplots(1, len(key_families), figsize=(18, 5))

for ax, fam in zip(axes, key_families):
    # step a: build the 6 monthly column names for this family
    # (e.g. for "daily_avg_bal" -> m1_daily_avg_bal, m2_daily_avg_bal, ..., m6_daily_avg_bal)
    cols = [f"m{m}_{fam}" for m in range(1, 7)]

    # step b: keep only those 6 columns + the target
    plot_df = train[cols + [config.TARGET]].copy()

    # step c: go from "wide" (6 columns, one per month) to "long" (1 "month" column + 1 "value" column)
    # this is needed because seaborn.lineplot expects an x column and a y column, not 6 separate columns
    plot_df = plot_df.melt(id_vars=config.TARGET, value_vars=cols, var_name="month", value_name="value")

    # step d: extract the month number from the column name. IMPORTANT (confirmed in
    # data_dictionary.csv): m1 is the MOST RECENT month and m6 is the OLDEST month, so this
    # number is really "how many months ago" -- not a normal forward-counting month index.
    plot_df["months_ago"] = plot_df["month"].str.extract(r"m(\d)_").astype(int)

    # step e: plot months_ago on the x-axis, then invert the axis. matplotlib draws ascending
    # values left-to-right by default, so without inverting, "1 month ago" (most recent) would
    # sit to the LEFT of "6 months ago" (oldest) -- i.e. the chart would read backwards in time.
    # invert_xaxis() flips that so the chart reads left-to-right as oldest -> most recent,
    # like a normal time series.
    sns.lineplot(data=plot_df, x="months_ago", y="value", hue=config.TARGET, estimator="mean", ax=ax)
    ax.invert_xaxis()
    ax.set_title(fam)
    ax.set_xlabel("months ago (6 = oldest -> 1 = most recent)")

plt.tight_layout()
plt.show()

# --- numeric readout: the numbers behind what the chart shows ---
# % change from 6 months ago (m6, oldest) to the most recent month (m1) -- i.e. the actual
# chronological trend as the customer approaches the prediction point. We use both mean AND
# median because these financial variables tend to have strong outliers that can distort
# the mean; if mean and median tell the same story, the pattern is more trustworthy.
print("% change m6 (6 months ago) -> m1 (most recent) by target group:\n")
for fam in key_families:
    m1_col, m6_col = f"m1_{fam}", f"m6_{fam}"
    for stat_name, stat_fn in [("mean", "mean"), ("median", "median")]:
        # mean (or median) of m1 and m6, computed separately for target=0 and target=1
        grouped = train.groupby(config.TARGET)[[m1_col, m6_col]].agg(stat_fn)
        # % change: (most_recent - oldest) / oldest * 100
        pct_change = (grouped[m1_col] - grouped[m6_col]) / grouped[m6_col] * 100
        t0 = pct_change.get(0, float("nan"))
        t1 = pct_change.get(1, float("nan"))
        print(f"{fam:25s} [{stat_name:8s}] target=0: {t0:+7.1f}%   target=1: {t1:+7.1f}%")
    print()